In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm_api
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle

In [49]:
################ Load data ready for nested logit
df = pd.read_csv('data_with_IV.csv')

# Generate nesting_id
df['nesting_ids'] = pd.factorize(df['fuel_type'])[0]

# Generate log of charging station stock
df['log_charging_IV'] = np.log(df['charging_stations_stock_lag']) 

# Create indicator for electric vehicles
df['is_electric'] = df['type'].apply(lambda x: 1 if x == '国产新能源乘用车' else 0)

# Make year an object variable
df['year'] = df['year'].astype(str)

# Rename columns to match pyblp requirements
df.rename(columns={
    'product_id': 'product_ids',
    'province_id': 'province_ids',
    'market_id': 'market_ids',
    'weighted_Avg_Price': 'prices',
    'market_share': 'shares'
}, inplace=True)

# calculate number of products in each market x nest
df['product_set_size'] = df.groupby(['market_ids', 'nesting_ids'])['product_ids'].transform('count')



In [50]:
################ Prepare policy data net prices of policies
# Import the province policy data
province_policy = pd.read_excel(r'C:\Users\Lenovo\Desktop\Dissertaion\China Data\Demand\Policy\Province_policy.xlsx')

# Import national policy data
national_policy = pd.read_excel(r'C:\Users\Lenovo\Desktop\Dissertaion\China Data\Demand\Policy\National_policy.xlsx')

# Sort province data
def expand_year_range(row):
    years = []
    if pd.notnull(row['year']):
        parts = str(row['year']).split('-')
        if len(parts) == 2:
            start, end = int(parts[0]), int(parts[1])
            years = list(range(start, end + 1))
        else:
            years = [int(parts[0])]
    return years

# Expand year
expanded_rows = []
for idx, row in province_policy.iterrows():
    for y in expand_year_range(row):
        new_row = row.copy()
        new_row['active_year'] = y
        expanded_rows.append(new_row)

province_policy_panel = pd.DataFrame(expanded_rows)

# Drop 'year' 
province_policy_panel.drop(columns=['year'], inplace=True)

# Rename columns to match the demand policy DataFrame
province_policy_panel.rename(columns={'active_year': 'year'}, inplace=True)
# Extract provinces and years
province_list = df['province'].unique()
years = list(range(2019, 2024))

# Construct policy panel
demand_policy_df = pd.DataFrame([(province, year) for province in province_list for year in years],
                        columns=['province', 'year'])

# 将 year 转为整数并排序
demand_policy_df['year'] = demand_policy_df['year'].astype(int)
demand_policy_df = demand_policy_df.sort_values(['year', 'province']).reset_index(drop=True)

# 如果 df 也需要排序
df['year'] = df['year'].astype(int)
df = df.sort_values(['year', 'province']).reset_index(drop=True)

# Merge national policy data
demand_policy_df = demand_policy_df.merge(national_policy, on='year', how='left')

# Merge province policy data
demand_policy_df = demand_policy_df.merge(province_policy_panel, on=['province', 'year'], how='left')


In [51]:




# Fill all NaN values with 0
demand_policy_df.fillna(0, inplace=True)

# Convert subsidy to units of 10000 CNY
demand_policy_df['sub'] = demand_policy_df['sub'] / 10000

# Sort year of demand policy DataFrame
demand_policy_df['year'] = demand_policy_df['year'].astype(int)
demand_policy_df = demand_policy_df.sort_values(['year', 'province']).reset_index(drop=True)
df['year'] = df['year'].astype(int)
df = df.sort_values(['year', 'province']).reset_index(drop=True)

# Merge the demand policy DataFrame with the main DataFrame
df = df.merge(demand_policy_df, on=['province', 'year'], how='left')

In [55]:
demand_policy_df[demand_policy_df['province'] == '吉林省'].head()

,province,year,sub_PHEV,sub_BEV_1,sub_BEV_2,sub_BEV_3,Tax,nat,sub,inc
4,吉林省,2019,1.00,1.8,1.80,2.500,0.911,1.0,0.000025,0.0
35,吉林省,2020,0.85,0.0,1.62,2.225,0.911,1.0,0.000025,0.0
66,吉林省,2021,0.68,0.0,1.30,1.800,0.911,1.0,0.000025,0.0
97,吉林省,2022,0.48,0.0,0.91,1.260,0.911,1.0,0.000025,0.0
128,吉林省,2023,0.00,0.0,0.00,0.000,0.911,0.0,0.000000,0.0


In [56]:
################ Compute net prices of policies
# Define the function to compute net prices
def compute_net_price(share):
    rg = share['range']
    p = share['prices']
    t = share['Tax']
    sub = share['sub']
    nat = share['nat']
    PHEV = share['sub_PHEV'] if share['fuel_type'] == 'PHEV' else 0
    BEV1 = share['sub_BEV_1'] if share['fuel_type'] == 'BEV' else 0
    BEV2 = share['sub_BEV_2'] if share['fuel_type'] == 'BEV' else 0
    BEV3 = share['sub_BEV_3'] if share['fuel_type'] == 'BEV' else 0
    def range_category(r, BEV1, BEV2, BEV3):
        if 250 <=r < 300: 
            return BEV1
        elif 300 <= r < 400:
            return BEV2
        elif 400 <= r:
            return BEV3
        else:
            return 0
        
    BEV = range_category(rg, BEV1, BEV2, BEV3)


    if share['nat'] == 0:
        return p * t - sub - PHEV - BEV

    elif share['nat'] == 1:
        return p * t - nat * 10000 * sub * (BEV + PHEV) - PHEV - BEV
    else:
        return 0

# Compute net prices
df['net_prices'] = df.apply(compute_net_price, axis=1)

# Export df as csv
df.to_csv('model_ready.csv', index=False)

In [4]:
################ Prepare supply-side data with charging policy
supply_df = pd.read_csv('supply_side_data.csv')
charging_policy_df = pd.read_excel(r"C:\Users\Lenovo\Desktop\Dissertaion\China Data\Charging\Charging_policy.xlsx")

def expand_year_range(row):
    years = []
    if pd.notnull(row['year']):
        parts = str(row['year']).split('-')
        if len(parts) == 2:
            start, end = int(parts[0]), int(parts[1])
            years = list(range(start, end + 1))
        else:
            years = [int(parts[0])]
    return years

# 展开年份区间
expanded_rows = []
for idx, row in charging_policy_df.iterrows():
    for y in expand_year_range(row):
        new_row = row.copy()
        new_row['active_year'] = y
        expanded_rows.append(new_row)

charging_policy_df = pd.DataFrame(expanded_rows)

charging_policy_df = charging_policy_df[['province', 'active_year', 'sub_fix', 'sub_ope']]

# Rename active_year to year for consistency
charging_policy_df.rename(columns={'active_year': 'year'}, inplace=True)

# Merge charging policy data with supply-side data
supply_df = supply_df.merge(charging_policy_df, on=['province', 'year'], how='left')
supply_df['sub_fix'] = supply_df['sub_fix'].fillna(0)
supply_df['sub_ope'] = supply_df['sub_ope'].fillna(0)

# Export supply_df as csv
supply_df.to_csv('supply_ready.csv', index=False)